# Logistic Regression Code Companion

This notebook connects Logistic Regression code with the theory: linear score, sigmoid probability, decision boundary, classification prediction, and classification metrics.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    RocCurveDisplay
)


## Load a Classification Dataset

Logistic Regression predicts probabilities for classes. This dataset has two classes, so it is a binary classification problem.


In [ ]:
cancer = load_breast_cancer(as_frame=True)
X = cancer.data
y = cancer.target

print("Classes:", dict(enumerate(cancer.target_names)))
X.head()


In [ ]:
print("Rows and features:", X.shape)
print("Target counts:")
print(y.value_counts().rename(index=dict(enumerate(cancer.target_names))))


## Split the Data

Training data is used to learn the parameters. Testing data is kept unseen until evaluation.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


## Train the Model

Logistic Regression first calculates a linear score $z$, then converts it to a probability using the sigmoid function. Scaling helps optimization because the features have different units and ranges.


In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(max_iter=1000, random_state=42))
])

model.fit(X_train, y_train)


## Predict Classes and Probabilities

`predict_proba()` gives probabilities. `predict()` applies the default decision boundary of 0.5 and returns the final class.


In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

results = pd.DataFrame({
    "actual": y_test.map(dict(enumerate(cancer.target_names))),
    "predicted": pd.Series(y_pred, index=y_test.index).map(dict(enumerate(cancer.target_names))),
    "probability_class_1": y_proba
})

results.head(10)


## Evaluate the Classifier

The confusion matrix shows correct and incorrect predictions. Accuracy, Precision, Recall, F1-Score, and ROC-AUC each answer a different evaluation question.


In [ ]:
cm = confusion_matrix(y_test, y_pred)
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1_score": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_proba)
}

print("Confusion matrix:")
print(cm)
print()
for name, value in metrics.items():
    print(f"{name}: {value:.3f}")


In [ ]:
print(classification_report(y_test, y_pred, target_names=cancer.target_names))


## Visualize the ROC Curve

The ROC curve shows how the classifier behaves across many decision boundaries, not only 0.5.


In [ ]:
RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title("Logistic Regression ROC Curve")
plt.tight_layout()
plt.show()
